<a href="https://colab.research.google.com/github/Prakum14/MLOps/blob/main/MPI_Distributed_Matrix_Multiplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- 1. Install and import essentials ---
!sudo apt-get update -qq
!sudo apt-get install -y openmpi-bin libopenmpi-dev
!pip install MPI
!pip install -q mpi4py numpy

import subprocess
import os
import re
import pandas as pd

# --- 2. MPI Python script content ---
mpi_script_template = """
import sys
import time
import numpy as np
from mpi4py import MPI

MATRIX_DIM = {MATRIX_DIM_PLACEHOLDER}

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

def create_matrix(rows, cols, seed=None):
    if seed is not None:
        np.random.seed(seed)
    return np.random.rand(rows, cols) * 10

def serial_mult(A, B):
    start = time.time()
    C = np.dot(A, B)
    end = time.time()
    return C, end - start

def distributed_mult(dim, comm, rank, size):
    A = B = None
    if rank == 0:
        A = create_matrix(dim, dim, seed=0)
        B = create_matrix(dim, dim, seed=1)
        start_time = time.time()
    else:
        start_time = 0

    B = comm.bcast(B, root=0)
    rows_per_process = dim // size
    if rank == 0:
        chunks = [A[i*rows_per_process:(i+1)*rows_per_process] for i in range(size-1)]
        chunks.append(A[(size-1)*rows_per_process:])
    else:
        chunks = None

    local_A = comm.scatter(chunks, root=0)
    local_C = np.dot(local_A, B)
    gathered = comm.gather(local_C, root=0)

    if rank == 0:
        C = np.vstack(gathered)
        total_time = time.time() - start_time
        return C, total_time
    return None, 0

if __name__ == "__main__":
    if rank == 0:
        A = create_matrix(MATRIX_DIM, MATRIX_DIM, seed=0)
        B = create_matrix(MATRIX_DIM, MATRIX_DIM, seed=1)
        _, serial_time = serial_mult(A, B)
        C, distributed_time = distributed_mult(MATRIX_DIM, comm, rank, size)
        speedup = serial_time / distributed_time if distributed_time > 0 else 0
        efficiency = speedup / size if size > 0 else 0

        print(f"Serial Time: {serial_time:.4f}")
        print(f"Distributed Time: {distributed_time:.4f}")
        print(f"Speedup: {speedup:.2f}")
        print(f"Efficiency: {efficiency:.2f}")
    else:
        distributed_mult(MATRIX_DIM, comm, rank, size)
    comm.Barrier()
"""

# --- 3. Run function to execute and extract results ---
def run_mpi_experiment(num_processes, matrix_dim=100):
    script_file = "temp_mpi_script.py"
    content = mpi_script_template.replace("{MATRIX_DIM_PLACEHOLDER}", str(matrix_dim))
    with open(script_file, "w") as f:
        f.write(content)

    command = [
        "mpiexec", "--allow-run-as-root", "--oversubscribe",
        "-n", str(num_processes), "python", "-u", script_file
    ]

    result = subprocess.run(command, capture_output=True, text=True)
    os.remove(script_file)

    output = result.stdout
    print(f"➡️ Output for {num_processes} process(es):\n{output}")

    # Extract metrics using regex
    try:
        serial = float(re.search(r"Serial Time:\s*([\d.]+)", output).group(1))
        dist = float(re.search(r"Distributed Time:\s*([\d.]+)", output).group(1))
        speed = float(re.search(r"Speedup:\s*([\d.]+)", output).group(1))
        eff = float(re.search(r"Efficiency:\s*([\d.]+)", output).group(1))
        return {"Processes": num_processes, "Serial Time (s)": serial,
                "Distributed Time (s)": dist, "Speedup": speed, "Efficiency": eff}
    except Exception as e:
        print(f"⚠️ Could not parse output for {num_processes} processes.")
        return {"Processes": num_processes, "Serial Time (s)": None,
                "Distributed Time (s)": None, "Speedup": None, "Efficiency": None}

# --- 4. Run and summarize ---
matrix_size = 100
process_counts = [1, 2, 4, 8]

results = []
for count in process_counts:
    res = run_mpi_experiment(num_processes=count, matrix_dim=matrix_size)
    results.append(res)

# --- 5. Display results as table ---
df = pd.DataFrame(results)
print("\n📊 Summary Table:")
display(df)


^C
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
openmpi-bin is already the newest version (4.1.2-2ubuntu1).
openmpi-bin set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.3/466.3 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
➡️ Output for 1 process(es):
Serial Time: 0.0079
Distributed Time: 0.0009
Speedup: 8.96
Efficiency: 8.96

➡️ Output for 2 process(es):
Serial Time: 0.0156
Distributed Time: 0.0063
Speedup: 2.48
Efficiency: 1.24

➡️ Output for 4 process(es):
Serial Time: 0.0003
Distributed Time: 0.0090
Speedup: 0.03
Efficiency: 0.01

➡️ Output for 8 process(es):
Serial Time: 0.0119
Distributed Time: 0.0079
Speedup: 1.52
Efficienc

,Processes,Serial Time (s),Distributed Time (s),Speedup,Efficiency
0,1,0.0079,0.0009,8.96,8.96
1,2,0.0156,0.0063,2.48,1.24
2,4,0.0003,0.0090,0.03,0.01
3,8,0.0119,0.0079,1.52,0.19
